<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/intermediate/02_supervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Supervised Learning Algorithms

Compare the major supervised learning algorithms side-by-side on the same dataset. Understand when to use each.

**Algorithms:** Logistic Regression, Decision Trees, Random Forest, Gradient Boosting (XGBoost), SVM, KNN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# Load real dataset
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Dataset: {data.target_names} — {X.shape[0]} samples, {X.shape[1]} features')
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Algorithm Comparison

In [ ]:
# Define models as pipelines (scaler + model)
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'Decision Tree':       Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5))]),
    'Random Forest':       Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, n_jobs=-1))]),
    'Gradient Boosting':   Pipeline([('scaler', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=100))]),
    'SVM (RBF kernel)':    Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True))]),
    'KNN (k=5)':           Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
}

# Cross-validated evaluation
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    results[name] = {'mean': scores.mean(), 'std': scores.std()}

results_df = pd.DataFrame(results).T.sort_values('mean', ascending=False)
print('5-Fold Cross-Validation AUC-ROC:')
for name, row in results_df.iterrows():
    print(f"  {name:<25} {row['mean']:.4f} ± {row['std']:.4f}")

In [ ]:
# Train all models and plot ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax_roc = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

for (name, model), color in zip(models.items(), colors):
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, color=color, lw=1.5, label=f'{name} ({roc_auc:.3f})')

ax_roc.plot([0,1],[0,1],'k--', label='Random (0.500)')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.set_title('ROC Curves — All Models')
ax_roc.legend(fontsize=8); ax_roc.grid(True, alpha=0.3)

# Bar chart comparison
ax_bar = axes[1]
bars = ax_bar.barh(results_df.index, results_df['mean'], xerr=results_df['std'],
                    color='steelblue', alpha=0.8, capsize=5)
ax_bar.set_xlabel('AUC-ROC (5-fold CV)'); ax_bar.set_title('Model Comparison (mean ± std)')
ax_bar.set_xlim(0.85, 1.01); ax_bar.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Decision Boundaries (2D Visualization)

In [ ]:
# Use 2D dataset to visualize decision boundaries
X_2d, y_2d = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                   n_clusters_per_class=1, random_state=42)

models_2d = {
    'Logistic Reg': Pipeline([('s', StandardScaler()), ('clf', LogisticRegression())]),
    'Decision Tree': Pipeline([('s', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=4))]),
    'Random Forest': Pipeline([('s', StandardScaler()), ('clf', RandomForestClassifier(100))]),
    'SVM (RBF)':     Pipeline([('s', StandardScaler()), ('clf', SVC())]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
h = 0.02
x_min, x_max = X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5
y_min, y_max = X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X_2d, y_2d)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_2d[:,0], X_2d[:,1], c=y_2d, cmap='RdBu', s=20, alpha=0.8, edgecolor='k', lw=0.5)
    score = cross_val_score(model, X_2d, y_2d, cv=5).mean()
    ax.set_title(f'{name}\nCV acc={score:.3f}')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.suptitle('Decision Boundaries (2D)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Algorithm Selection Guide

| Algorithm | Pros | Cons | Best For |
|-----------|------|------|----------|
| Logistic Regression | Fast, interpretable, probabilistic | Linear boundary only | Baseline, explainability needed |
| Decision Tree | Interpretable, handles mixed types | Overfits easily | Quick insights, rule extraction |
| Random Forest | Robust, handles missing values | Slow, black box | Tabular data, general purpose |
| Gradient Boosting | Best performance on tabular data | Slow training, many hyperparams | Competitions, production tabular |
| SVM | Works in high dimensions | Slow for large n, needs scaling | Text classification, small datasets |
| KNN | Simple, non-parametric | Slow prediction, high memory | Low-dimensional, small datasets |